# ImageJev endpoint test

Use this notebook in Google Colab to test a deployed ImageJev API. It tests the health endpoint, one decision, batch decisions, and multi-question inference. Keep your API key private.

In [ ]:
import base64
import getpass
import json
from pathlib import Path

import requests

BASE_URL = input('ImageJev base URL, for example https://your-service.example.com: ').strip().rstrip('/')
API_KEY = getpass.getpass('API key (leave blank if authentication is disabled): ').strip()
HEADERS = {'X-API-Key': API_KEY} if API_KEY else {}
TIMEOUT = 120

if not BASE_URL.startswith(('http://', 'https://')):
    raise ValueError('BASE_URL must start with http:// or https://')

def show_response(response):
    print(f'{response.request.method} {response.url}')
    print(f'Status: {response.status_code}')
    try:
        body = response.json()
        print(json.dumps(body, indent=2))
        return body
    except ValueError:
        print(response.text)
        return response.text


In [ ]:
# 1. Health check
health_response = requests.get(f'{BASE_URL}/health', headers=HEADERS, timeout=TIMEOUT)
health = show_response(health_response)
health_response.raise_for_status()

## Test with a public image URL

The server downloads the image URL. Use a public image URL reachable by the deployed service.

In [ ]:
IMAGE_URL = 'https://images.unsplash.com/photo-1542291026-7eec264c27ff?w=1200'
decision_payload = {
    'image': IMAGE_URL,
    'decision': 'Is this a clear product photograph suitable for listing?',
    'schema': {
        'type': 'choice',
        'options': {
            'yes': 'a clear, well-lit, in-focus product photograph',
            'no': 'a blurry, dark, cluttered, or damaged product photograph',
        },
    },
}
response = requests.post(f'{BASE_URL}/decide', json=decision_payload, headers=HEADERS, timeout=TIMEOUT)
single_result = show_response(response)
response.raise_for_status()

In [ ]:
# 2. Batch test
batch_payload = {
    'images': [IMAGE_URL, IMAGE_URL],
    'decision': decision_payload['decision'],
    'schema': decision_payload['schema'],
}
response = requests.post(f'{BASE_URL}/decide_batch', json=batch_payload, headers=HEADERS, timeout=TIMEOUT)
batch_result = show_response(response)
response.raise_for_status()

In [ ]:
# 3. Multi-question test: one image, multiple schemas
multi_payload = {
    'image': IMAGE_URL,
    'questions': {
        'is_product': {
            'type': 'noul',
            'true': 'a product photograph',
            'false': 'a photograph that is not a product photograph',
        },
        'quality': {
            'type': 'score',
            'levels': ['poor, blurry, or dark', 'acceptable and usable', 'clear professional product photography'],
        },
    },
}
response = requests.post(f'{BASE_URL}/v1/systemone', json=multi_payload, headers=HEADERS, timeout=TIMEOUT)
multi_result = show_response(response)
response.raise_for_status()

## Test a local image upload

Run the next cell only if your server exposes `/decide_upload`. Colab will open a file picker.

In [ ]:
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))
upload_schema = json.dumps(decision_payload['schema'])
with open(filename, 'rb') as image_file:
    response = requests.post(
        f'{BASE_URL}/decide_upload',
        files={'file': (filename, image_file, 'application/octet-stream')},
        data={'decision': decision_payload['decision'], 'schema': upload_schema},
        headers=HEADERS,
        timeout=TIMEOUT,
    )
upload_result = show_response(response)
response.raise_for_status()

## Expected endpoint paths

- `GET /health`
- `POST /decide`
- `POST /decide_batch`
- `POST /v1/systemone`
- `POST /decide_upload`

If authentication is enabled, the notebook sends the key in `X-API-Key`. Do not print or commit the key. A 401 means the key or header is invalid; a 422 usually means the request schema is invalid; a 429 means the server rate limit was reached.